# AI-Powered Flood Risk Prediction
# LSTM
## Ghale

A sequence model for Murray Bridge flood risk. An LSTM reads the last 14 days of river
level and predicts whether the next day is a high-risk day. It uses the same label and the
same chronological split as `common.py`, and is scored with `common.evaluate` so the five
metrics line up with the other models.

In [1]:
!pip install tensorflow-cpu -q


[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\kingn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Load data

In [2]:
import numpy as np
import common
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, matthews_corrcoef
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

df = common.load_data()
threshold = common.risk_threshold(df)
levels = df["water_level_m"].to_numpy(dtype="float32")
labels = (levels >= threshold).astype("int32")
print("Rows:", len(df), "| threshold (m):", round(threshold, 3), "| positive rate:", round(float(labels.mean()), 3))

Rows: 6364 | threshold (m): 0.806 | positive rate: 0.2


## 2. Build sequences

Each sample is the previous 14 days of level (all before the target day, so no leakage), and
the target is whether the next day is high risk. `y_prev` is yesterday's label, kept for the
persistence baseline.

In [3]:
N = 14
X, y, y_prev = [], [], []
for t in range(N, len(levels)):
    X.append(levels[t - N:t])
    y.append(labels[t])
    y_prev.append(labels[t - 1])
X = np.array(X)
y = np.array(y)
y_prev = np.array(y_prev)
print("Sequences:", X.shape, "| target positive rate:", round(float(y.mean()), 3))

Sequences: (6350, 14) | target positive rate: 0.201


## 3. Chronological split and scaling

Same 70 / 15 / 15 split as `common.py`, oldest first. The scaler is fit on the training part
only.

In [4]:
i_val = int(len(X) * 0.70)
i_test = int(len(X) * 0.85)
X_tr, X_va, X_te = X[:i_val], X[i_val:i_test], X[i_test:]
y_tr, y_va, y_te = y[:i_val], y[i_val:i_test], y[i_test:]
yprev_te = y_prev[i_test:]

scaler = StandardScaler().fit(X_tr.reshape(-1, 1))


def prep(a):
    return scaler.transform(a.reshape(-1, 1)).reshape(a.shape[0], N, 1)


X_tr, X_va, X_te = prep(X_tr), prep(X_va), prep(X_te)
print("Train:", X_tr.shape, "| Val:", X_va.shape, "| Test:", X_te.shape)

Train: (4445, 14, 1) | Val: (952, 14, 1) | Test: (953, 14, 1)


## 4. Train the LSTM

A small LSTM with class weighting for the imbalance and early stopping on the validation loss.

In [5]:
w = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr)
class_weight = {0: float(w[0]), 1: float(w[1])}

model = Sequential([
    Input((N, 1)),
    LSTM(32),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy")
es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=50, batch_size=64,
          class_weight=class_weight, callbacks=[es], verbose=2)

Epoch 1/50


70/70 - 2s - 24ms/step - loss: 0.5738 - val_loss: 0.4760


Epoch 2/50


70/70 - 0s - 5ms/step - loss: 0.3524 - val_loss: 0.4060


Epoch 3/50


...

70/70 - 0s - 4ms/step - loss: 0.2827 - val_loss: 0.3415


Epoch 36/50


70/70 - 0s - 4ms/step - loss: 0.2801 - val_loss: 0.3490


Epoch 37/50


70/70 - 0s - 4ms/step - loss: 0.2780 - val_loss: 0.3432


## 5. Tune the decision threshold and evaluate

The LSTM outputs a probability, and 0.50 is not the best cut-off. We pick the threshold that
maximises F1 on the **validation** set (never the test set), then score the test set with the
five metrics against the persistence baseline.

In [6]:
prob_va = model.predict(X_va, verbose=0).ravel()
prob_te = model.predict(X_te, verbose=0).ravel()

grid = np.linspace(0.10, 0.90, 81)
threshold_star = float(grid[int(np.argmax([f1_score(y_va, (prob_va >= t).astype(int)) for t in grid]))])
pred_te = (prob_te >= threshold_star).astype(int)
print("Decision threshold (tuned on validation):", round(threshold_star, 3))

baseline = {"model": "Persistence baseline",
            "F1": f1_score(y_te, yprev_te),
            "MCC": matthews_corrcoef(y_te, yprev_te)}
common.comparison_table([common.evaluate("LSTM", y_te, pred_te, prob_te), baseline])

Decision threshold (tuned on validation): 0.72


,F1,MCC,RMSE,Brier,NSE
model,,,,,
LSTM,0.804,0.741,0.338,0.114,0.371
Persistence baseline,0.796,0.732,NaN,NaN,NaN


## 6. Save the model

The Keras model plus the scaler, the window length and the tuned threshold, so the dashboard
and the SHAP step rebuild the exact same inputs.

In [7]:
repo = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
mdir = repo / "backend" / "models"
mdir.mkdir(parents=True, exist_ok=True)
model.save(mdir / "lstm.keras")
joblib.dump({"scaler": scaler, "window": N, "threshold": round(threshold_star, 3)}, mdir / "lstm_scaler.joblib")
print("Saved backend/models/lstm.keras and lstm_scaler.joblib (threshold %.3f)" % threshold_star)

Saved backend/models/lstm.keras and lstm_scaler.joblib (threshold 0.720)
